In [1]:
# =============================================================================
# 1. ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
"""
Initialize the analysis environment by loading essential modules and setting up
the Python path to access custom analysis functions.
"""

# Enable automatic reloading of modules for interactive development
%load_ext autoreload
%autoreload 2

# Import essential system modules
import sys
from pathlib import Path

# Define the path to custom analysis modules
# Note: Update this path to match your local installation
MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

# Add module path to system path for importing custom functions
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")
print("🔄 Auto-reload enabled for interactive development")

✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis
🔄 Auto-reload enabled for interactive development


In [2]:
# ALM recordings — sessions of interest
sessions = [
    "ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38",
    "ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35",
    "ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43",
]


In [5]:
# Build CD zarrs for the listed sessions
from pathlib import Path

from ephys_dimension_reduction_CD_pipeline import build_cd_dataset
from general_utils import smart_read_csv

# --- Config ---
binsize = "0.1"
align = "trial_start"
brain_regions_groups = [
   # ["MOs2/3", "MOs5", "MOs6a"],  # ALM
    [],                            # all units
]
time_windows = [[0, 1], [-1, 0]]
trial_types = ["right_choice_trials", "left_choice_trials"]

psth_root = Path("/root/capsule/scratch/psth_results")
behavior_root = Path("/root/capsule/scratch/behavior_summary")
cd_root = Path("/root/capsule/scratch/CD_results")
#metadata_path = Path("/root/capsule/scratch/qc_passed_units_metadata_all_sessions.csv")

#metadata = smart_read_csv(metadata_path)

failed = build_cd_dataset(
    sessions=sessions,
    psth_root=psth_root,
    behavior_root=behavior_root,
    cd_root=cd_root,
    #metadata=metadata,
    binsize=binsize,
    align=align,
    brain_regions_groups=brain_regions_groups,
    time_windows=time_windows,
    trial_types=trial_types,
    projection_time_window=None,
    two_fold_cv=True,
    norm_mode="divide_sqrtN",
    min_units_num=30,
    overwrite=True,
)



Session: ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38
  Region RG_ALL (ALL): ALL units
    ❌ Error in time window [0, 1]: Unknown align='trial_start'
    ❌ Error in time window [-1, 0]: Unknown align='trial_start'

Session: ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35
  Region RG_ALL (ALL): ALL units
    ❌ Error in time window [0, 1]: Unknown align='trial_start'
    ❌ Error in time window [-1, 0]: Unknown align='trial_start'

Session: ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43
  Region RG_ALL (ALL): ALL units
    ❌ Error in time window [0, 1]: Unknown align='trial_start'
    ❌ Error in time window [-1, 0]: Unknown align='trial_start'

All sessions done.
Failed items:
 - ('ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38', 'RG_ALL', 'time_window=[0, 1]', "Unknown align='trial_start'")
 - ('ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38', 'RG_ALL', 'time_window=[-1, 0]', "Unknown align='trial_start'")
 - ('

In [ ]:
# Per-session CD plots (one figure set per session)
import os

from ephys_dimension_reduction_CD_pipeline import (
    cd_save_path, load_cd_session, plot_cd_session, region_label,
)

cd_root = "/root/capsule/scratch/CD_results"
behavior_root = "/root/capsule/scratch/behavior_summary"

# Pick which region group + time-window combo to visualize per session.
# Must match one of the combos produced by the build cell.
viz_region_group = []            # [] => all units (matches RG_ALL files)
viz_time_window = (-1, 0)
distribution_window = (-1, 0)

region_lbl, region_print = region_label(viz_region_group)
print(f"Viewing region: {region_lbl} ({region_print}), TW={viz_time_window}, align={align}")

for session in sessions:
    zpath = cd_save_path(
        cd_root, session, region_lbl, trial_types, viz_time_window, align=align,
    )
    beh_csv = os.path.join(behavior_root, f"behavior_summary-{session}.csv")
    if not zpath.exists():
        print(f"[skip] CD zarr not found: {zpath}")
        continue
    if not os.path.exists(beh_csv):
        print(f"[skip] behavior CSV not found for {session}")
        continue

    print(f"\n=== Session: {session} ===\nFile: {zpath.name}")
    sess = load_cd_session(zpath, beh_csv)
    plot_cd_session(sess, distribution_window=distribution_window)


In [ ]:
# Aggregated CD across the 3 sessions (uses viz_region_group / viz_time_window above)
import os

from ephys_dimension_reduction_CD_pipeline import (
    aggregate_cd_sessions, cd_save_path, load_cd_session,
    plot_cd_aggregate, region_label,
)

region_lbl, _ = region_label(viz_region_group)

sessions_data = []
for session in sessions:
    zpath = cd_save_path(
        cd_root, session, region_lbl, trial_types, viz_time_window, align=align,
    )
    beh_csv = os.path.join(behavior_root, f"behavior_summary-{session}.csv")
    if not zpath.exists() or not os.path.exists(beh_csv):
        print(f"[skip] {session}")
        continue
    sessions_data.append(load_cd_session(zpath, beh_csv))

agg = aggregate_cd_sessions(sessions_data)
if not agg.counts_df.empty:
    print(agg.counts_df)
plot_cd_aggregate(agg, distribution_window=distribution_window)
